# 14 — Cross-cohort stability: the article's figures

Figure 3 and Supplementary Figs. Sa–Sc for **one state set**, chosen in the next cell (HMM,
mean-shift or threshold; `tools/stability_sweep.py` compares them all). The last cell gives the
numbers the text cites.

* **Figure 3.** A.1 reference transition graph (every cohort and condition, each once) · A.2 movie
  vs rest in the same people · B every cohort (occupancy, p(j | i), dynamic metrics) · C the
  transition structure replicates across cohorts and conditions (r over the noise ceiling; with
  `BALANCE = "time"` every cohort contributes the same hours of scan per draw) · D which properties
  depend on context (|d| movie vs rest in the same people; < 0.2 stable, 0.2–0.5 moderate,
  > 0.5 context-dependent).
* **Sa–Sc.** Variation between films, film by film: dynamic metrics, occupancy, transitions.

In [ ]:
import sys
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from fmri_decomposition.unsupervised_model import cohort_tables as ct
from fmri_decomposition.figures import cohorts as cp

# ---- THE STATE SET: any atlas x window x clusterer stage 4b wrote (listed below)
ATLAS    = "harvardoxford"
WINDOW_S = 30.0
STATES   = "HMM2_umap3_8"          # e.g. "HMM2_pca3_8", "ThresholdCluster_umap3_8", "MeanShift_umap3_3"

# ---- panel C: data balance. "time" = every cohort the same HOURS of scan per draw (whole scans,
# pooled transition counts): people and stimulus duration balanced together. None = cohort means.
BALANCE  = "time"
HOURS    = 10
VERSUS   = "mean"                  # C: each cohort vs the mean of the others; "pairs" = every pair
N_BOOT   = 200                     # draws for panel C, bootstrap over people for panel D
PVALUE   = "stars"                 # Figure 3C p-values: None, "exact" or "stars" (* <.05, ** <.01, *** <.001; BH-FDR)
TEST     = "both"                  # "difference" (paired t), "tost" (equivalence at ±0.2 SD) or "both"
SAVE_DIR = None                    # e.g. "figures/14": PNG + SVG of every figure

cfg = ct.Settings(
    space=dict(atlas=ATLAS, window_s=WINDOW_S, states=STATES),
    entropy_estimator="miller_madow",
    distinct_minutes="min",   # distinct transitions rarefied to the shortest scan (None = as stored)
    cohorts={"ds002837": dict(dirs=["ds002837"], role="train"),
             "cneuromod": dict(dirs=["cneuromod"], role="train"),
             "camcan": dict(dirs=["camcan", "camcan_rest"], role="projected"),
             "hcp7t_fix": dict(dirs=["hcp7t_fix"], role="projected")},
    rest_pattern=r"rest\d*",          # Rest, REST1..4 (HCP); every other task is movie
    alpha=0.10,                        # BH-FDR (A.2, films)
    fc_task=1.0, fc_dyn=0.25,          # A.2: |log2FC| for the graph and the dynamics
    edge_top=12,                       # A.1/A.2: strongest arrows drawn
    sesoi_d=0.2,
    save_dir=SAVE_DIR,
)
sets = ct.available_sets(cfg.root, cfg.cohorts, cfg.modelling_version)
print(f"{len(sets)} state sets have every cohort; at {ATLAS}, {WINDOW_S:g} s:")
print(", ".join(sets[(sets.atlas == ATLAS) & (sets.window_s.astype(float) == WINDOW_S)]["states"]))
co = ct.Cohorts(cfg)
co.tasks()

## Figure 3

Rest and movie together: each person's mean of the conditions they have. A, every cohort; B, the
reference graph (the cohorts' mean, each cohort once); C, which properties depend on context (movie vs
rest in the same people).

In [ ]:
st = co.stability(n_boot=N_BOOT)
cp.fig_figure3(co, st, pvalue=PVALUE, test=TEST);


### Figure 3, alternative: one occupancy number

The same figure, with the occupancy of each state replaced in C by one per-person number: the correlation between their occupancy profile and the reference profile (as for the transition structure).

In [ ]:
cp.fig_figure3(co, st, occupancy="fit", name="figure3_occupancy_fit", pvalue=PVALUE, test=TEST)
print(f"thinnest arrow drawn in B: p = {co.diagram_p_min:.2f}");


### Check: does one number per family lose information?

Movie vs rest |d| for each family's profile (top row of each panel) and for every member: each transition, each state's occupancy, each dynamic metric. The dynamics profile scales each metric by its SD first; with 5 metrics it is coarse.

In [ ]:
_, st_detail = cp.fig_aggregation_check(co, n_boot=N_BOOT, pvalue=PVALUE, test=TEST)


## Supplementary Figs. Sa–Sd

Sa–Sc: one row per task (rest, then every film, episode or run), grouped by cohort. Filled = the
same people did several tasks of their cohort (HCP, CNeuroMod, Cam-CAN); hollow = each film had its own
viewers (NNDb). Dashed blue = the global mean (every cohort and condition, each once). Occupancy
and transitions on 0–1; the dynamic metrics on the same fixed ranges as Figure 3.
Sd: the transition structure replicates (`VERSUS`: each cohort vs the mean of the others, or every pair).

In [ ]:
fs = co.film_summary(rest=True)
DYN = ["transition_fit", *co.dyn]
cp.fig_films_forest(co, fs, DYN, cp.global_means(co, DYN),
                    title="Supplementary Fig. Sa — dynamic metrics, task by task", name="Sa_films_dynamics");


In [ ]:
cp.fig_films_forest(co, fs, co.occ, cp.global_means(co, co.occ),
                    title="Supplementary Fig. Sb — occupancy of every state, task by task", name="Sb_films_occupancy",
                    labels={o: f"S{o[4:]} (share of time)" for o in co.occ});


In [ ]:
cp.fig_films_matrices(co, fs);


In [ ]:
pairs, ceil = co.replication("movie", balance=BALANCE, hours=HOURS, n_boot=N_BOOT, versus=VERSUS)
cp.fig_replication(co, pairs);


## Check: does a metric follow scan length?

Spearman ρ, within each cohort, between each metric and the minutes of scan behind it (one value per person and task). Entropy is corrected (Miller–Madow) and distinct transitions are rarefied to `co.distinct_minutes`; a metric still flagged here (|ρ| ≥ 0.3, FDR < 0.05) needs a correction too. Where films differ in length (NNDb), length and content go together: a flag, not a proof.

In [ ]:
print(f"distinct transitions rarefied to {co.distinct_minutes} min")
dc = co.duration_check()
flag = dc[(dc.rho.abs() >= .3) & (dc.p_fdr < .05)]
display(dc.pivot_table(index="variable", columns=["cohort", "kind"], values="rho").round(2))
flag.round(3)

## Numbers for the text

Sd (replication over the noise ceiling, 95% CI over draws), and every variable on one |d|
scale with its tier: rest vs movie in the same people; film vs film in the same people (HCP,
CNeuroMod; rest left out); movie across cohorts (different people, films and scanners: "alike despite", not "why").

In [ ]:
display(pairs.round(3))
co.context_summary(st, fs).round(2)